<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

Your exercise for this learning experience will be to write an extended Kalman filter (EKF) for map-based localization.

# Defining the map

We need to input the true locations of the landmarks that we are going to use for localization. If you are running in the Duckiematrix, this is done for you. If you are running on the real robot, you will need to measure the positions of these landmarks and write them into [the map file](../../packages/ekf_localization/map.yaml). The format of the map file is as follows:

```yaml
map:
  "1":
    position: [0.6786, 1.76085]
  "20":
    position: [-0.02925, 1.8252]
  ...
```

The numbers e.g. "1" are the IDs of the april tags. You should also be careful to define origin of the world frame. In the Duckiematrix, the yellow duckie starts at the origin of the world frame pointing in the direction of positive 'y' with positive 'x' to the right.

You may need to [assemble some of your own traffic signs](https://docs.duckietown.com/ente/duckietown-manual/30-duckietown-city/assembly/traffic-signs/duckietown-smart-city-traffic-signs-assembly-instructions.html) to have enough for a good localization performance. 

# Defining the filter parameters

Your EKF requires certain [parameters to be defined](../../packages/ekf_localization/config/ekf_localization_node/default.yaml). These include the initial pose and covariance of the Duckiebot, as well as the process and measurement covariances. Again, if you are running in the Duckiematrix, you can probably leave these alone since we have already specified the initial robot pose to coincide with the location of the Duckiebot in the Duckiematrix. In the real Duckietown you will need to make sure that you initially place the Duckiebot at roughly the position that is specified in your config. 

**Note**: You should return the Duckiebot to the initial pose every time you test your code. In the Duckiematrix you can do this by pushing the `R` key.


# Implementing your EKF

All the code that you will have to write is in the [ekf.py](../../packages/solution/ekf.py) file (although if you are interested to see some of the details of how the [ROS node](../../packages/ekf_localization/src/ekf_localization_node.py) is implemented). 

The pose of the Duckiebot is stored in `self.q` and is a numpy array with the form
$$
[x, y, \theta].
$$

The state covariance of the Duckiebot is stored in `self.P` and has the structure

$$
P = 
\begin{bmatrix}
P_{xx} &  P_{xy} & P_{x\theta} \\
P_{xy} & P_{yy} & P_{y\theta} \\
p_{x\theta} & P_{y\theta} & P_{\theta\theta}
\end{bmatrix}.
$$

There are two functions to write: `predict` and `update`. 


## The prediction step

The predict function 

```python
def predict(self, dX, dT):
```

takes as input `dX`, which is the linear forward movement of the Duckiebot since the last time the function was called (in meters), and `dT`, which is the amount that the Duckiebot has turned since the last time the function was called (in radians). These values are calculated from the encoder data and act as a proxy for our input (control).

The process model covariance is stored in `self.Q` and has the structure

$$
Q = 
\begin{bmatrix}
Q_{xx} & 0 \\
0 & Q_{\theta\theta}
\end{bmatrix},
$$

where $Q_{xx}$ and $Q_{\theta\theta}$ are covariances specified [in the config file](../../packages/ekf_localization/config/ekf_localization_node/default.yaml).

The implementation of the predict function has three steps:

### Step 1: Propagate the state

```python
# Step 1: update the pose estimate using the kinematic model
# TODO: Update these equations
self.q[0] = self.q[0]
self.q[1] = self.q[1]
self.q[2] = self.q[2]
```
The first step is to propagate forward the state estimate using the noiseless motion model $f$. We can use our kinematics model for this. For more details you should refer to the [LX on kinematics and odometry](https://docs.duckietown.com/ente/duckietown-manual/60-learning-experiences/lx-setup-modeling-kinematics.html)

**Note**: Make sure that you call the `angle_wrap` function everytime you modify the orientation. We want the orientation to always be in $[-\pi,\pi]$.

### Step 2: Calculate the process model Jacobians

```python
# Step 2: Calculate the process model Jacobians
# TODO: Define F and W
F = np.array([])
W = np.array([])
```

Next we need to linearize the process model. To do so we need to calculate: $F = \frac{\partial f}{\partial x}$ and $W = \frac{\partial f}{\partial w}$. Since there are 3 equations in $f$ and three states in `self.q`, the $F$ matrix will be $3\times 3$. Since there are only two "control inputs" and we assume that the noise is additive on those control input the $W$ matrix will be $3\times 2$.

### Step 3: Update the state covariance

```python
# Step 3: update the state covariance estimate
# TODO: update this equation
self.P = self.P
```

Finally, we need to update our prediction of the state covariance using the Jacobians we just calculated as well as the process noise covariance.


## The update step

The measurements of the landmarks (April tags), have two components, a range (distance to the feature) and a bearing (angle to the feature). Note that these measurements are relative to the current robot pose since the camera is attached to the robot. 

The update function:

```python
    def update(self, z: np.ndarray, tag_xy: np.ndarray):
```

takes as input `z` which is 2D array containing $[r, \phi]$ where $r$ is the range and $\phi$ is the bearing, and `tag_xy` which contains a 2D position $[l_x, l_y]$ of the landmark that was detected (that we read from the [map file](../../packages/ekf_localization/map.yaml)).

### Step 1: Calculate the predicted measurements

```python
# Step 1: calculate the predicted range and bearing measurements
# TODO: update the equations below
rng_pred = 1.0
bearing_pred = 0.0
z_pred = np.array([rng_pred, bearing_pred])
```

The first step is to use our measurement model, `h` to calculate the (noiseless) predictions of the measurements. To do so we can use our current state estimate and the known tag position. In order to do so you will have to derive the measurement models. In other words, given that we know the current state of the robot and the landmark position, what should the range be? what should the bearing be? For the calculation of the bearing be particularly careful to calculate it **in the current robot frame**. 

### Step 2: Calculate the innovation

```python
# Step 2: Calculate the innovation
# TODO: Define y
y = np.array([0.0, 0.0])
```

The innovation is the difference between the measurements we actually received (`z`) and the ones that we predicted using our measurement model (`z_pred`).

### Step 3: Calculate the measurement model Jacobian

```python
# Step 3: Calculate the measurement Jacobian
# TODO: Define H
H = np.array([])
```

Next we need to linearize the measurement model. To do so we need to calculate: $H = \frac{\partial h}{\partial x}$. Since we have two measurements and three states this matrix will be $2\times 3$.

### Step 4: Calculate the Kalman gain

```python
# Step 4: Calculate the Kalman gain
# TODO: Define K
K = np.array([])
```

Next we calculate the Kalman gain, `K`, which is used to weight our previous estimates against the information provided by these new measurements. 


### Step 5: Update the pose and pose covariance estimates

```python
# Step 5: Update the state and covariance estimates
# TODO: update these equations
self.q = self.q
self.q[2] = wrap_angle(self.q[2])
self.P = self.P
```

Finally, use the innovation, Kalman gain, and measurement Jacobian to update the estimate for the pose and the associated covariance using the EKF equations. 


# Testing and debugging

Once you have completed the implementation you can proceed to test your code using the approach described in the [README](../../README.md). If your estimator is performing well, you should see that the ground truth state remains inside the covariance ellipse of the estimate that you are producing as you pilot your vehicle around (you can use the joystick in noVNC for this). If you are running in the Duckiematrix we can plot the ground truth state. If you are running on a real Duckiebot, you would have to do the estimate yourself. 

If you have made an error in your implementation, it is likely that your estimate will diverge very quickly. Debugging this can actually be quite challenging since the divergence happens quickly and it is difficult to determine the root cause. One thing that can help is to isolate the prediction and the update steps. Specifically, you can test each step independently by setting `no_update: True` or `no_predict: True` in the [config file](../../packages/ekf_localization/config/ekf_localization_node/default.yaml). Be somewhat careful with the `no_predict: True` case however, since you are getting no information about how the robot is moving and relying entirely on the landmark measurements. If you have some period of time without landmark measurements your estimate will not be updated. Therefore a recommended method to proceed if you find that your estimator is not performing well is to begin with `no_update: True` until you are confident that your predict step is correct and then set `no_update: False` (leaving `no_predict` as `False` the entire time).

If you still have difficulty understanding what is going wrong, you may find it helpful to print out the intermediate results from the steps above and verify that they are correct one by one. 

Good luck!